# Customer Churn Prediction — Model Training & Evaluation

This notebook trains and evaluates the churn models using the **single
fit-once / serve-once `Pipeline`** defined in `src/pipeline.py` — the same object
the Streamlit app uses for inference. Building feature engineering, encoding and
scaling *inside* the pipeline removes the train/serve skew that comes from
re-fitting encoders at prediction time.

**Flow:** load → clean → split → cross-validate → evaluate on a held-out test set →
inspect → (persist via `train_model.py`).

## 1. Setup & imports

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

from src.data_loader import load_telco_data, clean_data, validate_data
from src.feature_engineering import separate_features_and_target
from src.pipeline import build_pipeline, NUMERIC_FEATURES, CATEGORICAL_FEATURES

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
RANDOM_STATE = 42
print('Libraries imported')

## 2. Load & prepare data

`load_telco_data` rejects synthetic/placeholder data; `clean_data` dedups, trims and coerces `TotalCharges` to numeric (missing values are imputed *inside* the pipeline).

In [ ]:
df = clean_data(load_telco_data('../data/WA_Fn-UseC_-_Telco_Customer_Churn.csv'))
validate_data(df)

X, y = separate_features_and_target(df)
print('Shape:', df.shape)
print(f'Churn rate: {y.mean():.2%}')
y.value_counts()

## 3. Train / test split (stratified)

The test set is held out and only touched once, at the end.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print(f'Train: {len(X_train)}  |  Test: {len(X_test)}')

## 4. The modeling pipeline

`build_pipeline(estimator)` returns one object:

`FeatureEngineer  →  ColumnTransformer(impute+scale | impute+one-hot)  →  estimator`

`class_weight` handles the ~27% churn imbalance, and `OneHotEncoder(handle_unknown='ignore')`
makes single-row / unseen categories safe at inference time.

In [ ]:
candidates = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_leaf=4,
        class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE),
}

# Visualise the pipeline structure
build_pipeline(candidates['Logistic Regression'])

## 5. Model selection via cross-validation (train only — no test leakage)

We pick the model by 5-fold CV ROC-AUC on the **training** split, so the test set stays untouched.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_scores = {}
for name, est in candidates.items():
    scores = cross_val_score(build_pipeline(est), X_train, y_train,
                             cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_scores[name] = scores.mean()
    print(f'{name:22s} CV ROC-AUC = {scores.mean():.4f} (+/- {scores.std():.4f})')

best_name = max(cv_scores, key=cv_scores.get)
print('\nBest by CV:', best_name)

## 6. Fit & evaluate on the held-out test set

In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    return {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
        'roc_auc':   roc_auc_score(y_true, y_proba),
    }

results = {}
for name, est in candidates.items():
    pipe = build_pipeline(est).fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    results[name] = (pipe, compute_metrics(y_test, pipe.predict(X_test), proba))

comparison = pd.DataFrame({n: m for n, (p, m) in results.items()}).T
comparison.round(4)

In [ ]:
best_pipe = results[best_name][0]
print(classification_report(y_test, best_pipe.predict(X_test),
                            target_names=['No Churn', 'Churn'], digits=4))

## 7. Model comparison chart

In [ ]:
ax = comparison[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']].plot.bar(figsize=(12, 6))
ax.set_title('Model comparison (test set)', fontweight='bold')
ax.set_ylim(0, 1)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 8. Confusion matrix — best model

In [ ]:
cm = confusion_matrix(y_test, best_pipe.predict(X_test))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title(f'Confusion Matrix — {best_name}')
plt.show()

## 9. Permutation feature importance (works for any estimator)

Importance is measured per **raw** input feature, so it stays interpretable even though the pipeline one-hot-encodes internally.

In [ ]:
imp = permutation_importance(best_pipe, X_test, y_test, n_repeats=5,
                             random_state=RANDOM_STATE, scoring='roc_auc', n_jobs=-1)

importance = (pd.DataFrame({'feature': X_test.columns, 'importance': imp.importances_mean})
              .sort_values('importance', ascending=False)
              .head(15))

ax = importance.set_index('feature')['importance'].iloc[::-1].plot.barh(
    figsize=(10, 8), color='#3498db')
ax.set_title(f'Permutation importance — {best_name}', fontweight='bold')
ax.set_xlabel('Mean ROC-AUC drop when shuffled')
plt.tight_layout()
plt.show()
importance

## 10. Persisting the model

The **canonical, reproducible** training entry point is `train_model.py` at the project root:
it runs exactly this flow and saves `models/churn_pipeline.pkl` + `models/metadata.json`,
which the Streamlit app loads. From here you could persist manually with
`joblib.dump(best_pipe, '../models/churn_pipeline.pkl')`.

In [ ]:
print('Best model:', best_name)
print(f"Test ROC-AUC: {results[best_name][1]['roc_auc']:.4f} | "
      f"Recall: {results[best_name][1]['recall']:.4f}")
print('\nRun `python train_model.py` from the project root to (re)train and persist artifacts.')